### Step 1: Import Libraries

Before building the dashboard, we import every library the project needs. Each one has a specific job:

| Import | What it is | Why we use it in this dashboard |
|---|---|---|
| `threading` | Python's built-in module for working with threads | Dash runs several chart callbacks **at the same time** in separate threads. We use `threading.local()` to give each thread its own private copy of the current theme (light/dark), so charts drawn in parallel never pick up each other's colours. |
| `from urllib.parse import quote` | A built-in function that makes text safe to use inside a URL | The sun and moon icons on the light/dark toggle are SVG images embedded directly in the CSS. `quote()` encodes the SVG code (spaces, `<`, `>`, `#` and so on) so the browser can read it as an image. |
| `numpy as np` | The standard library for fast numerical calculations | Used to create new columns with conditions, e.g. `np.where()` to calculate the **Response Gap** only for rows where an accident happened, and to label vehicles as **"Over limit"** or **"Within limit"**. |
| `pandas as pd` | The main library for working with tables (DataFrames) | Loads the cleaned CSV, converts dates, creates bands (congestion, age, experience) with `pd.cut()`, filters data, and calculates accident rates with `groupby()`. |
| `plotly.express as px` | Plotly's high-level charting library, which builds a chart in one line | Creates almost every chart in the dashboard: bar, line, box, pie, heatmap (`px.imshow`), sunburst and treemap. |
| `plotly.graph_objects as go` | Plotly's low-level charting library, with full control over every element | Used to build an **empty placeholder chart** with a "No data for this selection" message when the filters return no rows. |
| `Dash` | The main class that creates the web application | `app = Dash(__name__)` creates the dashboard app that runs in the browser. |
| `dcc` (Dash Core Components) | Interactive components | Dropdown filters, radio buttons, checkboxes, tabs, charts (`dcc.Graph`), the CSV download (`dcc.Download`) and data storage (`dcc.Store`). |
| `html` | Dash's version of HTML tags | Builds the page structure: `html.Div` (containers/cards), `html.H2` (title), `html.P` (text), `html.Button` (buttons). |
| `dash_table` | Dash's interactive table component | Displays the **"Top 10 longest closures"** table on the Closures tab. |
| `Input`, `Output`, `State` | The building blocks of **callbacks** (Dash's interactivity) | **Input**: a component whose change triggers an update (e.g. picking a year in a filter). **Output**: the component that gets updated (e.g. a chart). **State**: a value that is read but doesn't trigger an update by itself (e.g. the current filters, read only when the Download button is clicked). |

**In short:** `pandas` and `numpy` prepare the data, `plotly` draws the charts, and `dash` turns them into an interactive web dashboard. `threading` and `urllib` support the light/dark theme feature.

In [1]:
import threading
!pip install dash
from urllib.parse import quote

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, dash_table, Input, Output, State

#### Loading the dataset

In [2]:
DATA_PATH = "/content/synthetic_traffic_benchmark_200k_cleaned.csv"

### Step 2: Prepare New Columns

This step loads our cleaned dataset and creates the extra columns the dashboard needs. All of this runs **once**, when the app starts.

#### 2.1 Load the data and make a working copy

In [3]:
raw = pd.read_csv(DATA_PATH)
df = raw.copy()

#### 2.2 Convert text to real dates

In [4]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["Date"] = pd.to_datetime(df["Date"])

#### 2.3 Fixed orders for days, severity and months

In [5]:
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
SEV_ORDER = ["Minor", "Moderate", "Severe", "Fatal"]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
df["Day_of_Week"] = pd.Categorical(df["Day_of_Week"], categories=DAY_ORDER, ordered=True)

By default, Python sorts text **alphabetically**, which would put "Friday" before "Monday" and "Fatal" before "Minor". These lists define the natural order:

- `DAY_ORDER` keeps the heatmap rows in weekday order (Monday → Sunday).
- `SEV_ORDER` keeps severity charts ordered from least to most serious (Minor → Fatal).
- `MONTHS` turns month numbers (1–12) into readable labels (Jan–Dec) on the monthly response-gap chart.

`pd.Categorical(..., ordered=True)` tells pandas that `Day_of_Week` has a fixed order, so charts and groupings always follow Monday → Sunday.

#### 2.4 New calculated columns (feature engineering)

**Response Gap: how late was the emergency response?**

In [6]:
df["Response_Gap_min"] = np.where(
    df["Accident_Occurred"] == 1,
    df["Actual_Response_Time_min"] - df["Baseline_Expected_Response_Time_min"], np.nan)

- **Formula:** Response Gap = Actual response time − Expected response time
- Example: expected 10 min, actual 22 min → gap = **+12 minutes late**
- `np.where(condition, value_if_true, value_if_false)` only calculates the gap where an accident happened (`Accident_Occurred == 1`). Rows with no accident get `np.nan` (empty), because no emergency service was sent.
- Used in: **Tab 4 (Emergency response)** and the **Avg response gap** KPI card.

**Closure Hours: minutes converted to hours**

In [7]:
df["Closure_Hours"] = df["Road_Closure_Duration_min"] / 60

Road closure time is recorded in minutes. Dividing by 60 converts it to hours, which is easier to read (e.g. "16 hours" instead of "960 minutes").
Used in: **Tab 5 (Closures)** and the **Closure hours** KPI card.

**Speed Status: was the vehicle speeding?**

In [8]:
df["Speed_Status"] = np.where(df["Recorded_Speed_kmh"] > df["Speed_Limit_kmh"],
                              "Over limit", "Within limit")

Compares each vehicle's actual speed with the road's speed limit:
- Speed 72 km/h on a 60 km/h road → **"Over limit"**
- Speed 45 km/h on a 60 km/h road → **"Within limit"**

Used in: **Tab 3**, the "speed paradox" chart.

#### 2.5 Grouping numbers into bands with `pd.cut()`

Numbers like congestion (0–100) or driver age (16–85) have too many unique values to chart clearly. `pd.cut()` groups them into a few meaningful **bands** (bins).

**How `pd.cut()` works:** you give it the **bin edges** and a **label** for each bin. Each bin includes its right edge. For example, `[15, 24]` means "above 15, up to and including 24", i.e. ages 16–24.

| New column | Bin edges | Labels | Meaning |
|---|---|---|---|
| `Congestion_Band` | 0, 33, 66, 100 | Low, Medium, High | 0–33 = empty road, 34–66 = normal traffic, 67–100 = traffic jam |
| `Age_Group` | 15, 24, 34, 49, 64, 100 | <25, 25-34, 35-49, 50-64, 65+ | Driver age groups |
| `Experience_Band` | -1, 2, 5, 10, 60 | <2 yrs, 2-5 yrs, 5-10 yrs, 10+ yrs | Years of driving experience |

Two small details:
- `include_lowest=True` makes sure a congestion value of exactly **0** is included in the "Low" band.
- The experience bins start at **-1** so that drivers with **0 years** of experience are included in "<2 yrs".

Used in: **Tab 3**, the congestion, age group and experience charts.

In [9]:
df["Congestion_Band"] = pd.cut(df["Congestion_Index"], [0, 33, 66, 100],
                               labels=["Low", "Medium", "High"], include_lowest=True)
df["Age_Group"] = pd.cut(df["Driver_Age"], [15, 24, 34, 49, 64, 100],
                         labels=["<25", "25-34", "35-49", "50-64", "65+"])
df["Experience_Band"] = pd.cut(df["Driver_Experience_yrs"], [-1, 2, 5, 10, 60],
                               labels=["<2 yrs", "2-5 yrs", "5-10 yrs", "10+ yrs"])

#### 2.6 List of years for the filter

In [10]:
YEARS = sorted(df["Year"].unique())

- `df["Year"].unique()` finds every distinct year in the data: 2022, 2023, 2024, 2025.
- `sorted()` puts them in order.
- This list fills the **Year dropdown filter**, and is also used for the **year-over-year (YoY)** comparison on the KPI cards (to check whether a previous year exists).

**Summary:** after this step, `df` contains the original cleaned data **plus 6 new columns** (`Response_Gap_min`, `Closure_Hours`, `Speed_Status`, `Congestion_Band`, `Age_Group`, `Experience_Band`) that directly support the five business questions.

### Step 3: Define the Colour Themes (Light and Dark Mode)

All colours used in the dashboard are defined in **one place**: the `THEMES` dictionary. It holds two complete colour sets, one for **light mode** and one for **dark mode**.

---

#### 3.1 How the structure works

- `THEMES` is a **dictionary of dictionaries**. The outer keys are the theme names (`"light"`, `"dark"`) and each value is a dictionary of colours.
- Each colour has a **name** (called a *token*) describing its **role**, not the colour itself. For example, `ink` means "main text colour", not "black".
- Both themes use **exactly the same token names**, only with different colour values. That's what makes theme switching work: a chart simply asks for `P()["ink"]` and gets black text in light mode or white text in dark mode, without the chart code changing.

**Why keep all colours in one place?**
1. **Consistency:** every chart uses the same palette, so the dashboard looks like one system.
2. **Easy maintenance:** to change a colour, edit it once here and it updates everywhere.
3. **No hard-coded colours** scattered through the code.

Colours are written as **hex codes** (e.g. `#2a78d6`), a standard way of describing colours with red, green and blue values.

In [11]:
THEMES = {
    "light": dict(
        page="#f4f3ef", surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781",
        grid="#e1e0d9", axis="#c3c2b7", hover="#ffffff",
        series=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"],
        seq=["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"],        # low -> high
        sev=["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"],                   # Minor -> Fatal
        div_mid="#f0efec", soft_bar="#86b6ef", critical="#d03b3b",
    ),
    "dark": dict(
        page="#0d0d0d", surface="#1a1a19", ink="#ffffff", ink2="#c3c2b7", muted="#898781",
        grid="#2c2c2a", axis="#383835", hover="#262624",
        series=["#3987e5", "#d95926", "#199e70", "#c98500", "#d55181", "#008300", "#9085e9", "#e66767"],
        seq=["#184f95", "#256abf", "#3987e5", "#6da7ec", "#b7d3f6"],        # low recedes, high glows
        sev=["#184f95", "#2a78d6", "#6da7ec", "#cde2fb"],
        div_mid="#383835", soft_bar="#256abf", critical="#d03b3b",
    ),
}

### Step 4: Theme Helpers and Chart Styling Functions

This cell holds small **helper functions** that every chart uses. Instead of repeating the same styling code in 20 charts, we write it once here and call it everywhere.

#### 4.1 `_local = threading.local()`: a private theme note for each thread

In [12]:
_local = threading.local()

When the page loads or the theme toggle is clicked, Dash redraws many charts **at the same time**, each in its own **thread** (a separate line of work running in parallel).

`threading.local()` creates a storage object where **each thread gets its own private copy** of any value saved in it. So when one chart saves "dark" and another saves "light", neither can overwrite the other.

**Why not use a normal variable?** A normal global variable is **shared** by all threads. If the heatmap saved "dark" and, a moment later, another chart saved "light", the heatmap could read the wrong value and be drawn in the wrong colours. `threading.local()` prevents this.

> **Analogy:** instead of one shared whiteboard that anyone can overwrite, each painter gets their own sticky note with the colour scheme on it.

It stores **only the theme name** (`"light"` or `"dark"`). It doesn't store or cache charts, and it doesn't make the dashboard faster. Its purpose is **correctness**.

#### 4.2 `use_theme(theme)`: save the current theme

In [13]:
def use_theme(theme):
    _local.theme = theme if theme in THEMES else "light"

- Called at the **start of every chart callback**, e.g. `use_theme("dark")`.
- Saves the theme name in this thread's private note (`_local.theme`).
- **Safety check:** if the value isn't a valid theme (not in `THEMES`), it falls back to `"light"`, so the app never crashes from a bad value.

#### 4.3 `P()`: get the current theme's colours

In [14]:
def P():
    return THEMES[getattr(_local, "theme", "light")]

- `getattr(_local, "theme", "light")` reads the theme saved by `use_theme()`. If nothing has been saved yet, it returns `"light"` by default instead of raising an error.
- `THEMES[...]` then returns that theme's full colour dictionary.

**Example:**
```python
use_theme("dark")
P()["ink"]    # -> "#ffffff"  (white text)
use_theme("light")
P()["ink"]    # -> "#0b0b0b"  (near-black text)
```
This is why chart code can simply write `P()["grid"]` or `P()["series"]` and always get the correct colour for the active theme.

#### 4.4 `sev_map()`: a fixed colour for each severity level

In [15]:
def sev_map():
    return {**dict(zip(SEV_ORDER, P()["sev"])), "(?)": P()["axis"]}

This builds a dictionary that tells Plotly **exactly which colour** to use for each severity level. Step by step (light mode):

1. `zip(SEV_ORDER, P()["sev"])` pairs each severity with a colour:
   `("Minor", "#86b6ef"), ("Moderate", "#3987e5"), ("Severe", "#1c5cab"), ("Fatal", "#0d366b")`
2. `dict(...)` turns the pairs into a dictionary.
3. `**` unpacks it into a new dictionary, and one extra entry is added: `"(?)": P()["axis"]`.

**Result:**
```python
{"Minor": "#86b6ef", "Moderate": "#3987e5", "Severe": "#1c5cab", "Fatal": "#0d366b", "(?)": "#c3c2b7"}
```

- It's passed to charts as `color_discrete_map=sev_map()`, so **"Fatal" is always the same colour** in the stacked bar, sunburst, box plot and treemap, instead of Plotly picking random colours.
- **`"(?)"`:** in the sunburst and treemap, the inner rings (e.g. "Snow" → "Icy") contain a **mix** of severities, and Plotly labels those mixed sections `"(?)"`. We give them a neutral grey so only the outer severity ring is coloured.
- It's a **function** rather than a fixed dictionary so it always returns the colours for the **current** theme.

#### 4.5 `style(fig, title, x, y, legend)`: the shared look for every chart

In [16]:
def style(fig, title, x=None, y=None, legend=True):
    t = P()
    fig.update_layout(
        title=dict(text=title, font=dict(size=15, color=t["ink"]), x=0, xanchor="left"),
        font=dict(family="Inter, Segoe UI, Arial", size=12, color=t["ink2"]),
        paper_bgcolor=t["surface"], plot_bgcolor=t["surface"],
        margin=dict(l=50, r=20, t=55, b=40), autosize=True, showlegend=legend,
        legend=dict(orientation="h", y=1.02, yanchor="bottom", x=1, xanchor="right", title=None),
        hoverlabel=dict(bgcolor=t["hover"], font_size=12, font_color=t["ink"]), bargap=0.3,
    )
    fig.update_xaxes(title=x, gridcolor=t["grid"], linecolor=t["axis"], zeroline=False)
    fig.update_yaxes(title=y, gridcolor=t["grid"], linecolor=t["axis"], zeroline=False)
    fig.update_coloraxes(colorbar_tickfont_color=t["ink2"], colorbar_title_font_color=t["ink2"])
    return fig

Every chart passes through this function, so all charts look **consistent** and follow the current theme.

**Parameters:**
| Parameter | Meaning |
|---|---|
| `fig` | The Plotly chart to style |
| `title` | Chart title text |
| `x`, `y` | Axis titles (optional) |
| `legend` | Show or hide the legend (default: show) |

**What it sets:**
| Setting | Purpose |
|---|---|
| `title=dict(..., x=0, xanchor="left")` | Title in the main text colour, **left-aligned**, size 15 |
| `font=dict(family=..., color=t["ink2"])` | Default font and text colour for the whole chart |
| `paper_bgcolor`, `plot_bgcolor` | Chart background matches the card (`surface`), so the chart blends into its card |
| `margin=dict(l=50, r=20, t=55, b=40)` | Space around the chart for the title and axis labels |
| `autosize=True` | The chart fills its container (prevents empty gaps below charts) |
| `legend=dict(orientation="h", ... x=1, xanchor="right")` | Horizontal legend in the **top-right** corner |
| `hoverlabel=dict(...)` | Tooltip colours match the theme |
| `bargap=0.3` | Space between bars, so bars aren't too thick |
| `update_xaxes` / `update_yaxes` | Axis titles, theme-coloured gridlines and axis lines; `zeroline=False` removes the extra bold line at zero |
| `update_coloraxes(...)` | Makes the colour-bar text (e.g. on the heatmap) readable in both themes |

`t = P()` is called **once** at the top, so the function doesn't repeatedly look up the theme.

#### 4.6 `fit_labels(fig, horizontal)`: stop bar labels from being cut off

In [17]:
def fit_labels(fig, horizontal=False):
    """Leave room above (or right of) the longest bar so outside labels never get clipped."""
    vals = []
    for tr in fig.data:
        v = tr.x if horizontal else tr.y
        if v is not None:
            vals += [float(x) for x in v if x is not None]
    top = max(vals) if vals else 1
    fig.update_traces(cliponaxis=False, textfont_size=12)
    if horizontal:
        fig.update_xaxes(range=[0, top * 1.2])
    else:
        fig.update_yaxes(range=[0, top * 1.18])
    return fig

**The problem it solves:** value labels printed above bars (e.g. "7.93") were being **cut off** at the top of the tallest bar, because the bar reached the edge of the chart area.

**How it works:**
1. It loops through every trace (set of bars) in the chart and collects all the bar values.
   - Vertical bars: values are on the **y**-axis (`tr.y`).
   - Horizontal bars: values are on the **x**-axis (`tr.x`).
2. It finds the largest value (`top`). If there's no data, it uses 1 so the code doesn't crash.
3. `cliponaxis=False` allows labels to draw slightly outside the plot area instead of being hidden.
4. It sets the axis range to **18% more than the tallest bar** (20% for horizontal bars), leaving empty space for the labels.

**Example:** tallest bar = 7.93 → y-axis goes up to 7.93 × 1.18 ≈ **9.4**, leaving room for the "7.93" label.

#### 4.7 `empty_fig(msg)`: a friendly message when there's no data

In [18]:
def empty_fig(msg="No data for this selection"):
    fig = go.Figure()
    fig.add_annotation(text=msg, showarrow=False, font=dict(color=P()["muted"], size=14))
    fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
    return style(fig, "", legend=False)

If the user picks a filter combination with **no matching records** (e.g. a rare weather + road type + year), a chart would normally show an error or a broken empty grid. This function returns a **clean blank chart** instead:

- `go.Figure()` creates an empty chart.
- `add_annotation(...)` writes "No data for this selection" in the middle, in the muted text colour.
- Both axes are hidden, so no empty gridlines are drawn.
- It's passed through `style()` so the background still matches the current theme.

---

**Summary:**
| Function | Job |
|---|---|
| `_local`, `use_theme()`, `P()` | Store and read the current theme safely in each parallel thread |
| `sev_map()` | Consistent severity colours across all charts |
| `style()` | One shared, theme-aware look for every chart |
| `fit_labels()` | Keeps bar value labels fully visible |
| `empty_fig()` | Shows a clean message instead of a broken chart when filters return no data |

### Step 5: Page Styling (CSS) and the Light/Dark Toggle Design

Plotly charts are styled with Python (Step 4), but the **rest of the page** (background, cards, KPI tiles, tabs, dropdowns, buttons, table and the toggle switch) is styled with **CSS**, the language browsers use for colours, sizes and layout.

This cell builds all the CSS as Python text. It's later inserted into the page's `<style>` tag (through `app.index_string`, in the next step).

#### 5.1 `css_vars(t)`: turn a theme into CSS variables

In [19]:
def css_vars(t):
    return (f"--page:{t['page']};--surface:{t['surface']};--ink:{t['ink']};--ink2:{t['ink2']};"
            f"--muted:{t['muted']};--grid:{t['grid']};--axis:{t['axis']};--accent:{t['series'][0]};"
            # Dash 4 component variables (dropdowns, checklists, radio items)
            f"--Dash-Fill-Inverse-Strong:{t['surface']};--Dash-Text-Strong:{t['ink']};"
            f"--Dash-Text-Primary:{t['ink']};--Dash-Text-Weak:{t['ink2']};--Dash-Text-Disabled:{t['muted']};"
            f"--Dash-Stroke-Strong:{t['axis']};--Dash-Stroke-Weak:{t['grid']};"
            f"--Dash-Fill-Interactive-Strong:{t['series'][0]};--Dash-Fill-Interactive-Weak:{t['grid']};"
            f"--Dash-Shading-Weak:{t['grid']};--Dash-Fill-Disabled:{t['grid']};")

**What are CSS variables?** A CSS variable is a named value, written with two dashes, e.g. `--ink: #0b0b0b;`. Other CSS rules then use it with `var(--ink)`. Change the variable once and everything that uses it updates.

This function takes one theme dictionary from `THEMES` (Step 3) and converts it into a line of CSS variables:

```
--page:#f4f3ef; --surface:#fcfcfb; --ink:#0b0b0b; --accent:#2a78d6; ...
```

It creates two groups of variables:

| Group | Examples | Used by |
|---|---|---|
| **Our own variables** | `--page`, `--surface`, `--ink`, `--ink2`, `--muted`, `--grid`, `--axis`, `--accent` | Our CSS rules for the page, cards, KPI text, buttons, tabs and table |
| **Dash 4 component variables** | `--Dash-Text-Strong`, `--Dash-Stroke-Strong`, `--Dash-Fill-Interactive-Strong`, ... | Dash's **built-in** dropdowns, checkboxes and radio buttons, which read these specific names for their colours. Setting them makes Dash's own components follow our theme too. |

`--accent` uses the first series colour (blue). It's used for the Download button and the selected tab's top line.

#### 5.2 Sun and moon icons: `SUN_PATH`, `MOON_PATH`

In [20]:
SUN_PATH = ("M12 2.25a.75.75 0 0 1 .75.75v2.25a.75.75 0 0 1-1.5 0V3a.75.75 0 0 1 .75-.75ZM7.5 12a4.5 4.5 0 1 1 9 0 "
            "4.5 4.5 0 0 1-9 0ZM18.894 6.166a.75.75 0 0 0-1.06-1.06l-1.591 1.59a.75.75 0 1 0 1.06 1.061l1.591-1.59ZM21.75 "
            "12a.75.75 0 0 1-.75.75h-2.25a.75.75 0 0 1 0-1.5H21a.75.75 0 0 1 .75.75ZM17.834 18.894a.75.75 0 0 0 1.06-1.06l-1.59"
            "-1.591a.75.75 0 1 0-1.061 1.06l1.59 1.591ZM12 18a.75.75 0 0 1 .75.75V21a.75.75 0 0 1-1.5 0v-2.25A.75.75 0 0 1 12 "
            "18ZM7.758 17.303a.75.75 0 0 0-1.061-1.06l-1.591 1.59a.75.75 0 0 0 1.06 1.061l1.591-1.59ZM6 12a.75.75 0 0 1-.75.75H3"
            "a.75.75 0 0 1 0-1.5h2.25A.75.75 0 0 1 6 12ZM6.697 7.757a.75.75 0 0 0 1.06-1.06l-1.59-1.591a.75.75 0 0 0-1.061 1.06l1.59 1.591Z")
MOON_PATH = ("M9.528 1.718a.75.75 0 0 1 .162.819A8.97 8.97 0 0 0 9 6a9 9 0 0 0 9 9 8.97 8.97 0 0 0 3.463-.69.75.75 0 0 1 "
             ".981.98 10.503 10.503 0 0 1-9.694 6.46c-5.799 0-10.5-4.7-10.5-10.5 0-4.368 2.667-8.112 6.46-9.694a.75.75 0 0 1 .818.162Z")

```python
SUN_PATH = ("M12 2.25a.75.75 0 0 1 .75.75v2.25 ...")
MOON_PATH = ("M9.528 1.718a.75.75 0 0 1 .162.819 ...")
```

These long strings are **SVG path data**: drawing instructions that describe the shape of each icon (M = move to, a = draw an arc, v/h = vertical/horizontal line, Z = close the shape). They're the same sun and moon icons from the toggle design we were given (from the open-source **Heroicons** icon set).

#### 5.3 `svg_uri(path, color)`: turn an icon into an image the CSS can use

In [21]:
def svg_uri(path, color):
    svg = (f"<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24' fill='{color}'>"
           f"<path fill-rule='evenodd' clip-rule='evenodd' d='{path}'/></svg>")
    return "url(\"data:image/svg+xml," + quote(svg) + "\")"

1. It wraps the path in a complete **SVG image**, filled with the chosen colour (yellow for the sun, white for the moon).
2. `quote(svg)` **URL-encodes** the SVG, converting characters like `<`, `>`, `#` and spaces into a safe format.
3. It returns a **data URI**: `url("data:image/svg+xml,...")`. This lets the image be **embedded directly inside the CSS**, so no separate image files are needed.

**Why do it this way?** Dash has no component for writing raw SVG, and embedding the icons keeps the whole dashboard in **one `app.py` file**, with no extra image or CSS files to manage.

#### 5.4 `TOGGLE_CSS`: the animated light/dark switch

In [22]:
TOGGLE_CSS = f"""
.toggle {{ width: 72px; height: 36px; border-radius: 999px; border: none; padding: 0; cursor: pointer;
           position: relative; background: #F3F3F3; box-shadow: 2px 2px 6px rgba(0,0,0,.08) inset;
           transition: .5s; outline-offset: 3px; }}
.toggle-circle {{ width: 29px; height: 29px; border-radius: 99px; background: #fff; position: absolute;
           top: 50%; left: 0%; transform: translate(4px, -50%); overflow: hidden; transition: .5s;
           box-shadow: 2px 2px 4px rgba(0,0,0,.08), -2px -2px 6px rgba(0,0,0,.06) inset; }}
.toggle:active .toggle-circle {{ width: 36px; }}                       /* the little squish on press */
.toggle-icon {{ position: absolute; width: 16px; height: 16px; left: 50%; top: 50%;
           background-size: contain; background-repeat: no-repeat; transition: .5s; }}
.toggle-sun  {{ background-image: {svg_uri(SUN_PATH, '#FFD600')}; transform: translate(-50%, -50%); opacity: 1; }}
.toggle-moon {{ background-image: {svg_uri(MOON_PATH, '#FFFFFF')}; transform: translate(-50%, -200%); opacity: 0; }}
/* dark state: track darkens, circle slides right, sun drops out, moon drops in */
.theme-dark .toggle {{ background: #1F1F21; box-shadow: 2px 2px 6px rgba(0,0,0,.5) inset; }}
.theme-dark .toggle-circle {{ left: 100%; transform: translate(calc(-100% - 4px), -50%); background: #2C2C2F;
           box-shadow: 2px 2px 4px rgba(0,0,0,.5), -2px -2px 6px rgba(0,0,0,.5) inset; }}
.theme-dark .toggle-sun  {{ transform: translate(-50%, 100%); opacity: 0; }}
.theme-dark .toggle-moon {{ transform: translate(-50%, -50%); opacity: 1; }}
"""

This CSS recreates the toggle design we were given (a pill-shaped switch with a sliding circle), scaled down from 400×200 px to **72×36 px** so it fits the header.

| CSS class | What it styles |
|---|---|
| `.toggle` | The **pill-shaped track**: 72×36 px, fully rounded (`border-radius: 999px`), light grey with a soft inner shadow |
| `.toggle-circle` | The **white circle (knob)** that slides; starts on the left, centred vertically |
| `.toggle:active .toggle-circle` | While the button is being pressed, the circle **stretches** from 29 px to 36 px wide, a small "squish" effect |
| `.toggle-icon` | Shared settings for both icons: 16×16 px, centred inside the circle |
| `.toggle-sun` | Yellow sun: **visible** (`opacity: 1`) in the centre of the circle |
| `.toggle-moon` | White moon: **hidden** (`opacity: 0`) and moved above the circle |

**Dark state:** the rules starting with `.theme-dark` apply only when the page is in dark mode:

| Rule | Effect |
|---|---|
| `.theme-dark .toggle` | Track turns dark (`#1F1F21`) |
| `.theme-dark .toggle-circle` | Circle **slides to the right** (`left: 100%`) and turns dark grey |
| `.theme-dark .toggle-sun` | Sun **drops down and fades out** |
| `.theme-dark .toggle-moon` | Moon **drops into the centre and fades in** |

`transition: .5s` makes every change **animate smoothly over half a second** instead of jumping instantly.

#### 5.5 `CSS`: the full stylesheet for the page

In [23]:
CSS = TOGGLE_CSS + f"""
.theme-light {{ {css_vars(THEMES['light'])} color-scheme: light; }}
.theme-dark  {{ {css_vars(THEMES['dark'])}  color-scheme: dark; }}
html, body {{ margin: 0; }}
#root {{ background: var(--page); color: var(--ink); min-height: 100vh; padding: 20px 28px;
        font-family: Inter, 'Segoe UI', Arial, sans-serif; transition: background .2s, color .2s; }}
.card {{ background: var(--surface); border: 1px solid var(--grid); border-radius: 10px; padding: 8px; }}
.sub {{ color: var(--ink2); }}
.lbl {{ font-size: 12px; color: var(--ink2); font-weight: 600; }}
.kpi-title {{ font-size: 12px; color: var(--ink2); }}
.kpi-value {{ font-size: 26px; font-weight: 700; color: var(--ink); }}
.kpi-sub {{ font-size: 12px; color: var(--muted); }}
.btn {{ padding: 8px 14px; border-radius: 8px; border: 1px solid var(--axis); background: var(--surface);
        color: var(--ink); cursor: pointer; }}
.btn-primary {{ background: var(--accent); color: #fff; border: none; }}
/* tabs */
.tab {{ background: var(--page) !important; color: var(--ink2) !important;
        border: 1px solid var(--grid) !important; padding: 12px !important; }}
.tab--selected {{ background: var(--surface) !important; color: var(--ink) !important;
        border-top: 3px solid var(--accent) !important; font-weight: 600; }}
/* dropdown menus */
.dash-dropdown, .dash-dropdown * {{ background-color: var(--surface); color: var(--ink); }}
/* data table */
.dash-spreadsheet-container td, .dash-spreadsheet-container th {{
        background-color: var(--surface) !important; color: var(--ink) !important;
        border-color: var(--grid) !important; }}
.dash-spreadsheet-container th {{ background-color: var(--page) !important; }}
"""

The final stylesheet = the toggle CSS + the page CSS below.

**How the theme switch works:**
- `.theme-light { ... }` holds all the **light** CSS variables.
- `.theme-dark { ... }` holds all the **dark** CSS variables.
- The page's main container (`#root`) has **one** of these two classes. When the toggle is clicked, a callback swaps the class from `theme-light` to `theme-dark`, and **every element on the page instantly picks up the new colours**, because they all use `var(--...)`.
- `color-scheme: light / dark` tells the browser which mode to use for built-in elements such as scrollbars and checkboxes.

**Page element styles:**

| CSS rule | What it styles |
|---|---|
| `html, body { margin: 0; }` | Removes the browser's default white border around the page |
| `#root` | The whole page: background, text colour, padding, font, and a smooth 0.2 s colour transition when switching themes |
| `.card` | Every card (charts, KPI tiles, filter bar): background, thin border, rounded corners |
| `.sub` | Subtitle and hint text in the secondary colour |
| `.lbl` | Small bold labels above the filters ("Year", "Area type"...) |
| `.kpi-title` / `.kpi-value` / `.kpi-sub` | KPI card text: small title, **large bold number**, small grey YoY note |
| `.btn` / `.btn-primary` | Normal buttons (e.g. "Clear heatmap selection") and the blue primary "Download filtered data" button |
| `.tab` / `.tab--selected` | Normal tabs vs the **selected tab** (brighter background, bold text, blue line on top) |
| `.dash-dropdown` | Dropdown menus follow the theme's background and text colours |
| `.dash-spreadsheet-container td / th` | The Top-10 table's cells and header row follow the theme |

**Summary:**
| Part | Job |
|---|---|
| `css_vars()` | Converts a theme's colours into CSS variables, including the ones Dash's components use |
| `SUN_PATH`, `MOON_PATH`, `svg_uri()` | Build the toggle's sun/moon icons as images embedded in the CSS |
| `TOGGLE_CSS` | Styles and animates the pill-shaped light/dark switch |
| `CSS` | The full page stylesheet; switching one class (`theme-light` ↔ `theme-dark`) re-colours the entire page |

### Step 6: Create the Dash App and Layout Helpers

This cell creates the web application itself, attaches our CSS to the page, and defines the small **building blocks** used to assemble the dashboard layout.

In [24]:
app = Dash(__name__, title="Traffic Safety Dashboard", suppress_callback_exceptions=True)
server = app.server

| Part | Meaning |
|---|---|
| `Dash(__name__)` | Creates the dashboard application. `__name__` tells Dash where the script lives, so it can find any supporting files. |
| `title="Traffic Safety Dashboard"` | The text shown on the **browser tab**. |
| `suppress_callback_exceptions=True` | The dashboard uses **tabs**, and charts inside a tab that isn't open yet may not exist on the page. This setting stops Dash from raising errors about components that aren't displayed at that moment. |
| `server = app.server` | Dash runs on top of **Flask** (a Python web server). Exposing `server` lets the app be deployed to a hosting service later (e.g. Render or Heroku) with a production server like Gunicorn. Locally it isn't needed, but it's standard good practice. |

#### 6.2 Attach our CSS to the page: `app.index_string`

In [25]:
app.index_string = """<!DOCTYPE html>
<html><head>{%metas%}<title>{%title%}</title>{%favicon%}{%css%}<style>""" + CSS + """</style></head>
<body>{%app_entry%}<footer>{%config%}{%scripts%}{%renderer%}</footer></body></html>"""

Every web page is built from an **HTML template**. `index_string` lets us customise the template Dash uses, so we can insert our own **CSS** (built in Step 5) into the page's `<style>` tag.

The `{%...%}` parts are **placeholders** that Dash fills in automatically. They must all be present, or the app won't load:

| Placeholder | What Dash puts there |
|---|---|
| `{%metas%}` | Page settings (e.g. character encoding, mobile scaling) |
| `{%title%}` | The browser-tab title |
| `{%favicon%}` | The small icon on the browser tab |
| `{%css%}` | Dash's own built-in styles |
| `{%app_entry%}` | **Where our entire dashboard layout is inserted** |
| `{%config%}`, `{%scripts%}`, `{%renderer%}` | Dash's JavaScript that makes the page interactive |

The key line is `<style>""" + CSS + """</style>`, which inserts our full stylesheet (theme colours, cards, tabs, toggle switch) into the page. This keeps the whole dashboard in **one Python file**, with no separate `.css` file needed.

#### 6.3 Layout helper functions

These small functions are **reusable building blocks**. Without them, every chart would need about eight lines of styling code. With them, a whole row of two charts is **one line**:

In [26]:
def card(children):
    """
    Wraps any content in a white (or dark) rounded box - a "card".
    Used for every chart, KPI tile and the Top-10 table.

    children  -> whatever goes inside the box (a chart, text, a table...)
    className -> "card" links to the CSS rule that sets the background,
                 border and rounded corners. Because the colours come from CSS
                 variables, the card switches automatically in dark mode.
    """
    return html.Div(children, className="card")

**`card(children)`: a rounded box**
- Wraps any content (a chart, KPI tile or table) in a box with a background, border and rounded corners.
- `className="card"` links it to the `.card` CSS rule. Because that rule uses CSS variables, **cards switch to dark automatically** in dark mode.

In [27]:
def row(*children, cols=2):
    """
    Places cards side by side in a grid row.

    *children -> any number of cards, e.g. row(graph("a"), graph("b"))
                 (the * means "accept as many items as you pass in")
    cols      -> how many columns the row has; default is 2 (two charts side by side).
                 Use cols=1 for a full-width chart like the sunburst.
    """
    return html.Div(children, style={"display": "grid", "gap": "14px", "marginBottom": "14px",
                                     "gridTemplateColumns": f"repeat({cols}, minmax(0, 1fr))",
                                     "alignItems": "start"})

**`row(*children, cols=2)`: place cards side by side**
- `*children` accepts **any number** of cards (the `*` collects them all).
- Uses **CSS Grid** to split the row into `cols` equal columns (default 2; `cols=1` for a full-width chart).
- `gap: 14px` spaces the cards; `marginBottom: 14px` spaces the rows.
- `minmax(0, 1fr)` stops a wide chart from pushing its neighbour off screen.
- `alignItems: start` lets each card keep its own height instead of stretching to match a taller neighbour. **This removed the empty gap under the line chart.**

In [28]:
TALL = {"sunburst", "corr"}

def graph(id_):
    """
    Creates an empty chart placeholder inside a card.
    The callbacks later fill it with a figure, using the same id.

    id_ -> the chart's unique name, e.g. "heatmap". Callbacks use
           Output("heatmap", "figure") to send the chart to this spot.
    """
    h = "480px" if id_ in TALL else "360px"
    return card(dcc.Graph(id=id_, config={"displaylogo": False, "responsive": True},
                          style={"height": h}))

**`TALL` and `graph(id_)`: a chart placeholder**
- `TALL = {"sunburst", "corr"}` lists the charts that need more height. A **set** is used because checking membership (`id_ in TALL`) is fast and readable.
- `graph(id_)` creates an **empty chart slot** inside a card. It stays empty until a callback fills it.
- **The `id` connects layout and callbacks:** `graph("heatmap")` creates the slot, and a callback with `Output("heatmap", "figure")` sends the chart into it.
- Height: **480 px** for tall charts, **360 px** for the rest. The height is set on the container and the chart fills it exactly.
- `config`: `displaylogo: False` hides the Plotly logo; `responsive: True` resizes charts when the browser window changes size.

In [29]:
def dropdown(id_, label, options, placeholder="All"):
    """
    Creates one filter in the top filter bar: a small label plus a dropdown.
    Used for Year, Area type, Road type and Weather.

    id_         -> the filter's name; callbacks read it with Input("f-year", "value")
    label       -> the text shown above the dropdown, e.g. "Year"
    options     -> the choices, e.g. [2022, 2023, 2024, 2025]
    placeholder -> text shown when nothing is selected ("All" = no filter applied)
    """
    return html.Div([
        html.Label(label, className="lbl"),
        dcc.Dropdown(id=id_, options=[{"label": str(o), "value": o} for o in options],
                     multi=True, placeholder=placeholder),
    ], style={"flex": "1", "minWidth": "160px"})

**`dropdown(id_, label, options, placeholder)`: one filter**
- Creates a small bold label ("Year") above a dropdown menu.
- `options=[{"label": str(o), "value": o} for o in options]` converts a plain list (e.g. `[2022, 2023]`) into the format Dash expects: `label` is what the user sees, `value` is what the code receives. `str(o)` makes sure numbers display correctly as text.
- `multi=True` allows **several selections** (e.g. 2023 **and** 2024).
- `placeholder="All"` is shown when nothing is selected, meaning **no filter applied**.
- `flex: 1` makes all four filters share the width equally; `minWidth: 160px` stops them getting too narrow on small screens.

In [30]:
def tab(label, value, children):
    """
    Creates one tab (e.g. "1 · When & where") and the content shown when it's clicked.

    label    -> the text on the tab button
    value    -> the tab's internal id ("t1", "t2"...); dcc.Tabs(value="t1") opens tab 1 first
    children -> everything inside the tab: rows of charts, buttons, text
    """
    return dcc.Tab(label=label, value=value, className="tab", selected_className="tab--selected",
                   children=children)

**`tab(label, value, children)`: one dashboard tab**
- `label` is the text on the tab (e.g. "1 · When & where").
- `value` is the tab's internal id (`"t1"`, `"t2"`...), used to choose which tab opens first.
- `children` is everything shown inside that tab.
- `className="tab"` and `selected_className="tab--selected"` apply the theme-aware CSS: normal tabs are muted, and the **selected tab** is bold with a blue line on top.

**Summary:**
| Part | Job |
|---|---|
| `app = Dash(...)` | Creates the web app |
| `server = app.server` | Makes the app ready for deployment |
| `app.index_string` | Inserts our custom CSS into the page template |
| `card`, `row`, `graph`, `dropdown`, `tab` | Reusable building blocks that keep the layout code short, consistent and theme-aware. To change the look of every card or chart, edit one helper instead of 20 places. |

### Step 7: Data Helper Functions (Filtering and Accident Metrics)

This cell holds the functions every chart uses to **filter the data** and **calculate accident numbers**. Writing them once keeps all charts consistent: every chart filters and counts in exactly the same way.

#### 7.1 `filter_df(...)`: apply the dashboard filters

In [31]:
def filter_df(years, areas, roads, weathers):
    dff = df
    if years:    dff = dff[dff["Year"].isin(years)]
    if areas:    dff = dff[dff["Area_Type"].isin(areas)]
    if roads:    dff = dff[dff["Road_Type"].isin(roads)]
    if weathers: dff = dff[dff["Weather"].isin(weathers)]
    return dff

This takes the four values from the filter bar (Year, Area type, Road type, Weather) and returns only the matching rows.

**How it works:**
- `dff = df` starts with the full dataset. (`dff` is short for "filtered DataFrame".)
- Each `if` checks whether the user picked anything in that filter:
  - **Nothing selected** (the dropdown shows "All"): the value is empty (`None` or `[]`), which Python treats as `False`, so that filter is **skipped** and all values are kept.
  - **Something selected:** only matching rows are kept.
- `.isin(years)` keeps rows whose value is **in the selected list**. Because the dropdowns allow multiple selections, this handles one value or several, e.g. `[2023, 2024]` keeps both years.
- The filters are applied **one after another**, so they combine with **AND** logic.

**Example:**
| User selects | Rows returned |
|---|---|
| Nothing | All 199,403 records |
| Year = 2024 | Only 2024 records |
| Year = 2024, Area = Rural, Weather = Fog | Only **rural, foggy records from 2024** |

The original `df` is **never changed**. Filtering always creates a new, smaller table, so the next filter change starts again from the full data.

#### 7.2 `metric(dff, by, mode)`: accident rate or accident count

In [32]:
def metric(dff, by, mode):
    """Accident rate (%) or accident count grouped by `by`."""
    g = dff.groupby(by, observed=True)["Accident_Occurred"]
    out = (g.mean() * 100) if mode == "rate" else g.sum()
    return out.reset_index(name="value")

This calculates the main number shown on most charts, for each group (e.g. each road type or each hour).

| Parameter | Meaning | Example |
|---|---|---|
| `dff` | The filtered data | Output of `filter_df()` |
| `by` | Column(s) to group by | `"Road_Type"`, or `["Day_of_Week", "Hour"]` for the heatmap |
| `mode` | What to calculate | `"rate"` or `"count"`, from the **Rate / Count** switch |

**How it works:**
1. `groupby(by)` splits the data into groups, e.g. one group per road type.
2. `["Accident_Occurred"]` selects the accident column, which is **1** if an accident happened and **0** if not.
3. It then calculates:
   - **Rate mode:** `mean() × 100`. Because the column contains only 0s and 1s, its **average is the share of records that had an accident**. Multiplying by 100 turns it into a percentage.
   - **Count mode:** `sum()`, which adds up the 1s = **the total number of accidents**.
4. `reset_index(name="value")` turns the result back into a normal table with a column called `value`, ready for Plotly.

**Example (Interstate roads):**
- 37,937 records, of which 3,007 had an accident
- **Count** = 3,007
- **Rate** = 3,007 ÷ 37,937 × 100 = **7.93%**

**Why rate matters more than count:** busier road types have more records, so they naturally have more accidents. The **rate** gives a fair comparison, "how likely is an accident here?", which is why **Rate is the default**.

**`observed=True`:** some grouping columns (like `Congestion_Band` or `Day_of_Week`) are *categorical*. This setting tells pandas to only include categories that actually appear in the filtered data, avoiding empty groups and a pandas warning.

#### 7.3 `metric_label(mode)`: the matching axis title

In [33]:
def metric_label(mode):
    return "Accident rate (%)" if mode == "rate" else "Accidents (count)"

Returns the correct text for chart titles and axis labels, so they always match what's being shown:
- Rate mode → **"Accident rate (%)"**
- Count mode → **"Accidents (count)"**

#### 7.4 `FILTERS` and `THEME`: reusable callback inputs

In [34]:
FILTERS = [Input("f-year", "value"), Input("f-area", "value"),
           Input("f-road", "value"), Input("f-weather", "value")]
THEME = Input("theme", "data")

In Dash, an `Input` tells a callback: **"run again whenever this component changes."**

- `FILTERS` is a list of the **four filter dropdowns**. `Input("f-year", "value")` means "watch the `value` of the component with id `f-year`".
- `THEME` watches the **stored theme** (`"light"` or `"dark"`) saved in the `dcc.Store` with id `theme`, which the toggle button updates.

**Why store them in variables?** Almost every chart callback needs the same four filters plus the theme. Instead of retyping them in every callback, we write:

```python
@app.callback(Output("heatmap", "figure"), FILTERS + [Input("f-mode", "value"), THEME])
```

This keeps callbacks **short**, and guarantees every chart reacts to **exactly the same filters**. If a new filter were added later, it would only need adding in one place.

---

**Summary:**
| Function / variable | Job |
|---|---|
| `filter_df()` | Returns only the rows matching the selected filters ("All" = no filter) |
| `metric()` | Calculates **accident rate (%)** or **accident count** per group |
| `metric_label()` | Gives the matching title text for the chosen measure |
| `FILTERS`, `THEME` | Reusable callback inputs, so every chart updates when a filter or the theme changes |

### Step 8: Build the Dashboard Layout

This cell defines **what the dashboard looks like**: every section, filter, KPI card, tab and chart slot, arranged from top to bottom.

In Dash, the layout is a **tree of components**. Each component can contain other components through its `children`, like boxes inside boxes. At this stage the charts are **empty placeholders**. The callbacks (next step) fill them with data.

**Overall structure (top to bottom):**
```
root (whole page, holds the theme class)
├── 1. Header: title + subtitle + light/dark toggle
├── 2. Filter bar: 4 dropdowns + Rate/Count switch + Download button
├── 3. KPI cards: 5 summary numbers
└── 4. Tabs: one tab per business question
     ├── Tab 1: When & where
     ├── Tab 2: Weather & roads
     ├── Tab 3: Congestion vs speeding
     ├── Tab 4: Emergency response
     └── Tab 5: Closures & disruption
```

#### 8.1 KPI card lists

In [35]:
kpi_ids = ["kpi-records", "kpi-acc", "kpi-rate", "kpi-gap", "kpi-closure"]
kpi_titles = ["Records", "Accidents", "Accident rate", "Avg response gap", "Closure hours"]

Two matching lists: the **ids** (used by the callback to send each number to the right card) and the **titles** shown on the cards. Keeping them in lists lets us build all five cards with one short loop (see 8.5).

#### 8.2 The root container
- `app.layout` is the **entire page**.
- The outer `html.Div` with `id="root"` wraps everything.
- `className="theme-light"` is the **theme switch point**. When the toggle is clicked, a callback changes this to `"theme-dark"`, and the whole page instantly switches colours through the CSS variables (Step 5). The page **starts in light mode**.

#### 8.3 Header: title and theme toggle
| Component | Purpose |
|---|---|
| `display: flex` + `justifyContent: space-between` | Puts the title on the **left** and the toggle on the **right** |
| `html.H2` / `html.P` | The dashboard title and a one-line subtitle explaining its purpose |
| `html.Button(id="theme-btn", className="toggle")` | The **light/dark toggle switch**. It contains a circle (`toggle-circle`) holding the sun and moon icons. All the sliding and fading animation comes from the CSS in Step 5. |
| `n_clicks=0` | Counts button clicks; the callback reacts every time this number changes |
| `title="Switch light / dark mode"` | Tooltip shown when hovering over the toggle |
| `aria-label` | Describes the button for **screen readers** (accessibility), because the button shows icons and no text |
| `dcc.Store(id="theme", data="light")` | An **invisible storage box** in the browser that remembers the current theme. Clicking the toggle flips it between `"light"` and `"dark"`, and every chart callback reads it through the `THEME` input. |

#### 8.4 Filter bar
The filter bar is a **card** containing all the controls in one horizontal line. `flexWrap: wrap` moves items to a second line on small screens instead of squeezing them.

| Control | Details |
|---|---|
| **Year, Area type, Road type, Weather** | Built with the `dropdown()` helper (Step 6). The options come **directly from the data**, e.g. `sorted(df["Area_Type"].unique())` gives Rural, Suburban, Urban. If the data changed, the filters would update automatically. |
| **Measure (Rate / Count)** | `dcc.RadioItems(id="f-mode", value="rate")` switches charts between **accident rate (%)** and **accident count**. Rate is the default because it gives a fair comparison. |
| **Download filtered data** | A button (`id="btn-download"`) that exports the currently filtered rows as a CSV. |
| `dcc.Download(id="download")` | An invisible component that sends the CSV file to the user's browser when the button is clicked. |

#### 8.5 KPI cards
- A **grid of 5 equal columns**, one per KPI card.
- A **list comprehension** builds all five cards in one step: `zip(kpi_titles, kpi_ids)` pairs each title with its id, e.g. `("Records", "kpi-records")`.
- Each card has three lines:

| Line | Class | Example |
|---|---|---|
| Title | `kpi-title` | "Accident rate" |
| Value (filled by callback) | `kpi-value` | **4.12%** |
| Sub-text (filled by callback) | `kpi-sub` | "▲ 0.09 pts vs 2023" (the YoY comparison) |

The value and sub-text start **empty**. The KPI callback fills them using the ids (e.g. `kpi-rate` and `kpi-rate-sub`).

#### 8.6 Tabs: one per business question
`dcc.Tabs` holds five tabs, built with the `tab()` helper. `value="t1"` means **Tab 1 opens first**. Each tab uses the `row()` and `graph()` helpers to place its charts.

**Tab 1: When & where** (Question 1: when and where is accident risk highest?)
| Element | Purpose |
|---|---|
| Tip text | Tells the user they can click the heatmap to drill down |
| `row(graph("heatmap"), graph("hour-line"))` | Hour × day heatmap + accident rate by hour |
| `row(graph("road-bar"), graph("area-bar"))` | Accident rate by road type + by area type |
| "Clear heatmap selection" button | Resets the drill-down back to all data |

**Tab 2: Weather & roads** (Question 2: how do weather and road conditions change risk and severity?)
| Element | Purpose |
|---|---|
| `dcc.RadioItems(id="t2-factor")` | Switches the charts between **Weather** and **Road_Condition** |
| `row(graph("risk-mult"), graph("sev-stack"))` | Risk multiplier vs baseline + severity mix |
| `row(graph("sunburst"), cols=1)` | Full-width sunburst: weather → road condition → severity |

**Tab 3: Congestion vs speeding** (Question 3: does congestion, not speeding, drive accidents?)
| Element | Purpose |
|---|---|
| Two-column strip with an empty left cell | Places the checkbox **directly above the right-hand chart**, so both charts below still start at the same height |
| `dcc.Checklist(id="t3-control")` | "Control for congestion": splits the speed paradox chart by congestion level |
| `row(graph("cong-bar"), graph("speed-paradox"))` | Accident rate by congestion + the speed paradox chart |
| `row(graph("age-bar"), graph("exp-bar"))` | Accident rate by driver age group + by experience |
| `row(graph("corr"), cols=1)` | Full-width correlation heatmap |

**Tab 4: Emergency response** (Question 4: is emergency response meeting expectations?)
| Element | Purpose |
|---|---|
| `dcc.Dropdown(id="t4-dim")` | Chooses how the response-gap box plot is broken down: area type, weather, dispatch priority, severity or road type. `clearable=False` stops the user from leaving it empty. |
| `row(graph("gap-box"), graph("gap-weather"))` | Response gap box plot + average gap by weather |
| `row(graph("gap-month"), cols=1)` | Full-width monthly response gap, year over year |

**Tab 5: Closures & disruption** (Question 5: what is the operational cost of severe incidents?)
| Element | Purpose |
|---|---|
| `dcc.Checklist(id="t5-log")` | Switches the closure box plot to a **log scale**, useful because closure times have a long tail of extreme values |
| `row(graph("closure-box"), graph("collision-pie"))` | Closure duration by severity + collision-type share |
| `row(graph("treemap"), card([...DataTable...]))` | Closure-hours treemap + a **Top 10 longest closures** table for the outlier discussion |
| `dash_table.DataTable(id="top10", page_size=10)` | The table starts empty and is filled by the callback; `style_cell` and `style_header` set its font and spacing |

#### 8.7 How the layout connects to the callbacks

Every interactive component has a unique **`id`**. The callbacks use these ids to know **what to read** and **what to update**:

| Component id | Type | Used by callbacks as |
|---|---|---|
| `f-year`, `f-area`, `f-road`, `f-weather` | Filters | **Input**: re-draw charts when changed |
| `f-mode` | Rate / Count switch | **Input** |
| `theme-btn`, `theme` | Toggle + stored theme | **Input**: switch colours |
| `t2-factor`, `t3-control`, `t4-dim`, `t5-log` | Tab-specific controls | **Input** |
| `heatmap`, `hour-line`, `road-bar`, ... | Chart slots | **Output**: receive the finished charts |
| `kpi-records`, `kpi-rate`, ... | KPI cards | **Output**: receive the numbers |
| `btn-download` → `download` | Button → file | Click triggers the CSV export |

In [36]:
app.layout = html.Div(id="root", className="theme-light", children=[
    # ---- header + theme switch ----
    html.Div(style={"display": "flex", "justifyContent": "space-between", "alignItems": "flex-start"}, children=[
        html.Div([
            html.H2("Traffic Safety & Emergency Response Intelligence", style={"margin": "0"}),
            html.P("Where, when and why accidents happen, and how well we respond.",
                   className="sub", style={"marginTop": "4px"}),
        ]),
        html.Button(id="theme-btn", className="toggle", n_clicks=0, title="Switch light / dark mode",
                    **{"aria-label": "Switch light or dark mode"}, children=[
            html.Div(className="toggle-circle", children=[
                html.Span(className="toggle-icon toggle-sun"),
                html.Span(className="toggle-icon toggle-moon"),
            ]),
        ]),
        dcc.Store(id="theme", data="light"),
    ]),

    # ---- filter bar ----
    html.Div(className="card", style={"display": "flex", "gap": "14px", "flexWrap": "wrap",
                                      "alignItems": "flex-end", "padding": "12px",
                                      "marginBottom": "14px"}, children=[
        dropdown("f-year", "Year", YEARS),
        dropdown("f-area", "Area type", sorted(df["Area_Type"].unique())),
        dropdown("f-road", "Road type", sorted(df["Road_Type"].unique())),
        dropdown("f-weather", "Weather", sorted(df["Weather"].unique())),
        html.Div([
            html.Label("Measure", className="lbl"),
            dcc.RadioItems(id="f-mode", value="rate", inline=True,
                           options=[{"label": " Rate ", "value": "rate"},
                                    {"label": " Count ", "value": "count"}],
                           inputStyle={"marginLeft": "8px"}),
        ]),
        html.Button("Download filtered data", id="btn-download", className="btn btn-primary"),
        dcc.Download(id="download"),
    ]),

    # ---- KPI cards ----
    html.Div(style={"display": "grid", "gridTemplateColumns": "repeat(5, minmax(0,1fr))",
                    "gap": "14px", "marginBottom": "14px"}, children=[
        card([html.Div(t, className="kpi-title"),
              html.Div(id=i, className="kpi-value"),
              html.Div(id=i + "-sub", className="kpi-sub")])
        for t, i in zip(kpi_titles, kpi_ids)
    ]),

    # ---- tabs ----
    dcc.Tabs(id="tabs", value="t1", children=[
        tab("1 · When & where", "t1", [html.Br(),
            html.P("Tip: click a cell in the heatmap to drill the other charts into that day and hour.",
                   className="sub", style={"fontSize": "13px"}),
            row(graph("heatmap"), graph("hour-line")),
            row(graph("road-bar"), graph("area-bar")),
            html.Button("Clear heatmap selection", id="btn-clear", className="btn"),
        ]),
        tab("2 · Weather & roads", "t2", [html.Br(),
            dcc.RadioItems(id="t2-factor", value="Weather", inline=True,
                           options=["Weather", "Road_Condition"], inputStyle={"marginLeft": "10px"}),
            row(graph("risk-mult"), graph("sev-stack")),
            row(graph("sunburst"), cols=1),
        ]),
        tab("3 · Congestion vs speeding", "t3", [html.Br(),
            html.Div(style={"display": "grid", "gap": "14px", "marginBottom": "10px",
                            "gridTemplateColumns": "repeat(2, minmax(0, 1fr))"}, children=[
                html.Div(),                                   # empty left cell (over the congestion chart)
                dcc.Checklist(id="t3-control", value=[],      # right cell, directly over the speed paradox chart
                              options=[{"label": " Control for congestion (speed paradox chart)", "value": "on"}]),
            ]),
            row(graph("cong-bar"), graph("speed-paradox")),
            row(graph("age-bar"), graph("exp-bar")),
            row(graph("corr"), cols=1),
        ]),
        tab("4 · Emergency response", "t4", [html.Br(),
            html.Div([html.Label("Break response gap down by: "),
                      dcc.Dropdown(id="t4-dim", value="Area_Type", clearable=False,
                                   options=["Area_Type", "Weather", "Dispatch_Priority",
                                            "Accident_Severity", "Road_Type"],
                                   style={"width": "260px"})],
                     style={"display": "flex", "gap": "10px", "alignItems": "center"}),
            html.Br(),
            row(graph("gap-box"), graph("gap-weather")),
            row(graph("gap-month"), cols=1),
        ]),
        tab("5 · Closures & disruption", "t5", [html.Br(),
            dcc.Checklist(id="t5-log", value=[], options=[{"label": " Log scale", "value": "log"}]),
            row(graph("closure-box"), graph("collision-pie")),
            row(graph("treemap"), card([
                html.H4("Top 10 longest closures", style={"margin": "6px"}),
                dash_table.DataTable(id="top10", page_size=10,
                                     style_cell={"fontSize": "12px", "padding": "6px",
                                                 "fontFamily": "Inter, Arial"},
                                     style_header={"fontWeight": 700})])),
        ]),
    ]),
])

**Summary:** the layout is the dashboard's **skeleton**: a header with the theme toggle, a filter bar, five KPI cards, and five tabs, each built around one business question. Nothing is calculated here. Every chart and number is an **empty slot with an id**, filled in by the callbacks in the next step.

### Step 9: Callback Functions (Making the Dashboard Interactive)

Callbacks are what make the dashboard **interactive**. A callback is a Python function that Dash runs **automatically** whenever a user changes something (a filter, a checkbox, the theme toggle), and whose result updates part of the page.

**How every callback is structured:**
```python
@app.callback(Output(...), Input(...), State(...))
def function_name(input_values...):
    ...
    return new_value_for_output
```

| Part | Meaning |
|---|---|
| `@app.callback(...)` | A **decorator** that registers the function with Dash |
| `Output("id", "property")` | **What gets updated**, e.g. `Output("heatmap", "figure")` = the heatmap's chart |
| `Input("id", "property")` | **What triggers the update**; the function runs every time this value changes |
| `State("id", "property")` | A value that is **read but doesn't trigger** the callback by itself |
| Function arguments | Receive the Input/State values **in the same order** they're listed |
| `return` | The value(s) sent to the Output(s), in the same order |

**Common pattern in every chart callback:**
1. `use_theme(theme)` saves the current theme for this thread (Step 4).
2. `t = P()` gets the current theme's colours.
3. `filter_df(...)` applies the filter bar selections (Step 7).
4. Build the chart(s) with Plotly Express.
5. `style(...)` / `fit_labels(...)` apply the shared look (Step 4).
6. `return` the finished chart(s).

#### 9.1 Theme toggle: `flip_theme` and `set_theme`

In [37]:
@app.callback(Output("theme", "data"), Input("theme-btn", "n_clicks"), State("theme", "data"),
              prevent_initial_call=True)
def flip_theme(_, current):
    return "dark" if current == "light" else "light"

@app.callback(Output("root", "className"), THEME)
def set_theme(theme):
    return f"theme-{theme}"

- **Trigger:** clicking the toggle button changes its click count (`n_clicks`).
- **State:** reads the **current** theme from the `dcc.Store`.
- **Returns:** the opposite theme: light → dark, dark → light.
- `_` is used as the argument name because we don't need the click count itself, only the fact that a click happened.
- `prevent_initial_call=True` stops it running when the page first loads, so the dashboard **starts in light mode**.

```python
@app.callback(Output("root", "className"), THEME)
def set_theme(theme):
    return f"theme-{theme}"
```

- **Trigger:** the stored theme changes.
- **Returns:** `"theme-light"` or `"theme-dark"`, which becomes the page's main CSS class.
- This single class change **re-colours the entire page** (cards, filters, tabs, toggle animation) through the CSS variables in Step 5.

**The chain:** click toggle → `flip_theme` updates the store → `set_theme` changes the page class **and** every chart callback (which all have `THEME` as an input) redraws its chart in the new colours.

#### 9.2 KPI cards with year-over-year comparison: `update_kpis`

In [38]:
@app.callback([Output(i, "children") for i in kpi_ids] +
              [Output(i + "-sub", "children") for i in kpi_ids], FILTERS)
def update_kpis(years, areas, roads, weathers):
    dff = filter_df(years, areas, roads, weathers)
    acc = dff[dff["Accident_Occurred"] == 1]
    rate = dff["Accident_Occurred"].mean() * 100 if len(dff) else 0
    vals = [f"{len(dff):,}", f"{len(acc):,}", f"{rate:.2f}%",
            f"{acc['Response_Gap_min'].mean():.1f} min" if len(acc) else "-",
            f"{acc['Closure_Hours'].sum():,.0f} h"]

    subs = ["", "", "Select one year for YoY", "Actual minus baseline", ""]
    if years and len(years) == 1 and years[0] - 1 in YEARS:       # YoY comparison
        prev = filter_df([years[0] - 1], areas, roads, weathers)
        p_acc = prev[prev["Accident_Occurred"] == 1]
        p_rate = prev["Accident_Occurred"].mean() * 100
        d = lambda now, before: f"{'▲' if now >= before else '▼'} {abs(now - before) / before * 100:.1f}% vs {years[0] - 1}"
        subs[1] = d(len(acc), len(p_acc))
        subs[2] = f"{'▲' if rate >= p_rate else '▼'} {abs(rate - p_rate):.2f} pts vs {years[0] - 1}"
        subs[4] = d(acc["Closure_Hours"].sum(), p_acc["Closure_Hours"].sum())
    return vals + subs

- **Outputs:** 10 values in total: the 5 **main numbers** plus the 5 **sub-texts**, built with list comprehensions from `kpi_ids`.
- **Inputs:** the four filters. The theme isn't needed here because KPI text is coloured by CSS.

**Main values:**
| KPI | Calculation |
|---|---|
| Records | `len(dff)`: number of filtered rows |
| Accidents | `len(acc)`: rows where `Accident_Occurred == 1` |
| Accident rate | `mean() × 100` of the accident column |
| Avg response gap | Average `Response_Gap_min` of accidents (shows "-" if there are none) |
| Closure hours | Total `Closure_Hours` of accidents |

Formatting: `:,` adds thousand separators (199,403), `:.2f` gives 2 decimals (4.12%), `:,.0f` rounds to whole numbers.

**Year-over-year (YoY) comparison:**
```python
if years and len(years) == 1 and years[0] - 1 in YEARS:
```
YoY only runs when **exactly one year** is selected **and** the previous year exists in the data (e.g. 2024 selected → compare with 2023; 2022 has no previous year, so it's skipped).

- `prev = filter_df([years[0] - 1], ...)` gets the **previous year**, with the other filters kept the same.
- `d = lambda now, before: ...` is a small helper that returns **▲ or ▼** plus the **percentage change**, e.g. "▲ 3.5% vs 2023".
- **Accident rate** uses the change in **percentage points** ("▲ 0.09 pts"), because comparing two percentages as a percentage change would be confusing.
- Otherwise the sub-text shows a hint: *"Select one year for YoY"*.

#### 9.3 Tab 1: Heatmap: `t1_heatmap`

In [39]:
@app.callback(Output("heatmap", "figure"), FILTERS + [Input("f-mode", "value"), THEME])
def t1_heatmap(years, areas, roads, weathers, mode, theme):
    use_theme(theme)
    dff = filter_df(years, areas, roads, weathers)
    m = metric(dff, ["Day_of_Week", "Hour"], mode)
    grid = m.pivot(index="Day_of_Week", columns="Hour", values="value").reindex(DAY_ORDER)
    fig = px.imshow(grid, aspect="auto", color_continuous_scale=P()["seq"],
                    labels=dict(x="Hour of day", y="", color=metric_label(mode)))
    fig.update_traces(hovertemplate="%{y} %{x}:00<br>%{z:.2f}<extra></extra>")
    return style(fig, f"{metric_label(mode)} by hour and day", legend=False)

**Answers:** *At which day and hour is accident risk highest?*

1. `metric()` calculates the accident rate (or count) for every **day + hour** combination.
2. `pivot()` reshapes the result into a grid: **days as rows, hours (0–23) as columns**.
3. `reindex(DAY_ORDER)` puts the rows in weekday order (Monday → Sunday).
4. `px.imshow()` draws the grid as a **heatmap**, with darker/brighter cells for higher risk, using the theme's sequential colour scale.
5. `hovertemplate` sets the tooltip, e.g. *"Friday 17:00 – 6.21"*. `<extra></extra>` hides Plotly's extra trace-name box.

#### 9.4 Tab 1: Heatmap drill-down: `clear_click` and `t1_charts`

**Clear button:**

In [40]:
@app.callback(Output("heatmap", "clickData"), Input("btn-clear", "n_clicks"),
              prevent_initial_call=True)
def clear_click(_):
    return None

When a heatmap cell is clicked, Dash stores the click in `clickData`. The **"Clear heatmap selection"** button resets it to `None`, so the charts go back to showing all data.

**Linked charts:**

In [41]:
@app.callback([Output("hour-line", "figure"), Output("road-bar", "figure"), Output("area-bar", "figure")],
              FILTERS + [Input("f-mode", "value"), Input("heatmap", "clickData"), THEME])
def t1_charts(years, areas, roads, weathers, mode, click, theme):
    use_theme(theme)
    t = P()
    dff = filter_df(years, areas, roads, weathers)
    day = hour = None
    if click:
        pt = click["points"][0]
        day, hour = pt["y"], int(pt["x"])

    # line: hour profile (for the clicked day, if any)
    dline = dff[dff["Day_of_Week"] == day] if day else dff
    m = metric(dline, ["Hour", "Area_Type"], mode)
    line = px.line(m, x="Hour", y="value", color="Area_Type", markers=True,
                   color_discrete_sequence=t["series"])
    line.update_traces(line=dict(shape="spline", smoothing=1.3, width=2.5), marker_size=6,
                           hovertemplate="%{x}: %{y:.1f}")
    # line.update_traces(line_width=2, marker_size=6)
    if hour is not None:
        line.add_vline(x=hour, line_dash="dot", line_color=t["muted"])
    line = style(line, f"{metric_label(mode)} by hour" + (f" — {day}" if day else ""),
                 "Hour of day", metric_label(mode))

    # bars: road type + area type (for the clicked cell, if any)
    dcell = dff[(dff["Day_of_Week"] == day) & (dff["Hour"] == hour)] if click else dff
    suffix = f" — {day} {hour}:00" if click else ""
    overall = dcell["Accident_Occurred"].mean() * 100

    bars = []
    for col, title in [("Road_Type", "by road type"), ("Area_Type", "by area type")]:
        m = metric(dcell, col, mode).sort_values("value", ascending=False)
        fig = px.bar(m, x=col, y="value", color_discrete_sequence=[t["series"][0]],
                     text_auto=".2f" if mode == "rate" else ",")
        # value sits inside the top of each bar, so it never collides with the "Overall" line
        fig.update_traces(marker_cornerradius=4, textposition="inside", insidetextanchor="end",
                          textfont=dict(color="white", size=13), cliponaxis=False)
        if mode == "rate":
            fig.add_hline(y=overall, line_dash="dash", line_color=t["muted"], layer="below",
                          annotation_text=f"Overall {overall:.2f}%", annotation_position="top right",
                          annotation_font_color=t["muted"], annotation_bgcolor=t["surface"])
        bars.append(fit_labels(style(fig, f"{metric_label(mode)} {title}{suffix}", None,
                                     metric_label(mode), legend=False)))
    return line, bars[0], bars[1]

This is the dashboard's **cross-filtering** feature: clicking a heatmap cell updates the other three charts.

- **Reading the click:** `click["points"][0]` gives the clicked cell; `pt["y"]` is the **day** and `pt["x"]` is the **hour**.

**Line chart (accident rate by hour, one line per area type):**
- If a cell was clicked, it shows only **that day's** hourly pattern; otherwise all days.
- `line=dict(shape="spline", smoothing=1.3, width=2.5)` draws **smooth curved lines**.
- `add_vline(x=hour, line_dash="dot")` adds a dotted vertical line marking the **clicked hour**.
- The title adds the day, e.g. *"Accident rate (%) by hour — Friday"*.

**Bar charts (by road type and by area type):**
- If a cell was clicked, they use only records from **that exact day and hour**.
- A `for` loop builds both charts with the same code, avoiding duplication.
- `sort_values("value", ascending=False)` shows the highest bar first.
- `text_auto=".2f"` shows values on the bars (2 decimals for rate, thousand separators for count).
- `textposition="inside", insidetextanchor="end"` puts the value **inside the top of each bar** in white.
- **Overall average line** (rate mode only): `add_hline(y=overall, line_dash="dash", layer="below")` draws a dashed line at the overall rate, so bars above average stand out. `layer="below"` draws it **behind** the bars so it never crosses the labels.

#### 9.5 Tab 2: Weather and roads: `t2`

**Answers:** *How much does adverse weather increase accident risk and severity?*

- `factor` comes from the Weather / Road_Condition switch, so the **same code handles both** views.
- If there are no accidents in the filtered data, `empty_fig()` is returned for all three charts.

In [42]:
# ---- Tab 2: weather & roads ----
@app.callback([Output("risk-mult", "figure"), Output("sev-stack", "figure"), Output("sunburst", "figure")],
              FILTERS + [Input("t2-factor", "value"), THEME])
def t2(years, areas, roads, weathers, factor, theme):
    use_theme(theme)
    t = P()
    dff = filter_df(years, areas, roads, weathers)
    acc = dff[dff["Accident_Occurred"] == 1]
    if acc.empty:
        return empty_fig(), empty_fig(), empty_fig()

    base_cat = "Clear" if factor == "Weather" else "Dry"
    r = dff.groupby(factor)["Accident_Occurred"].mean()
    base = r.get(base_cat, np.nan)
    m = (r / base).sort_values().reset_index(name="multiple")
    m["rate"] = r.reindex(m[factor]).values * 100

    # chart 1: Bar chart
    mult = px.bar(m, x="multiple", y=factor, orientation="h", text=m["multiple"].map("{:.2f}×".format),
                  color_discrete_sequence=[t["series"][1]], custom_data=["rate"])
    mult.update_traces(marker_cornerradius=4, textposition="outside",
                       hovertemplate="%{y}: %{x:.2f}× baseline<br>Rate %{customdata[0]:.2f}%<extra></extra>")
    mult.add_vline(x=1, line_dash="dash", line_color=t["muted"],
                   annotation_text=f"{base_cat} baseline", annotation_position="top",
                   annotation_font_color=t["muted"])
    mult = fit_labels(style(mult, f"Accident risk vs {base_cat.lower()} baseline", "Risk multiple",
                            None, legend=False), horizontal=True)

    s = pd.crosstab(acc[factor], acc["Accident_Severity"], normalize="index").mul(100)
    s = s.reindex(columns=[c for c in SEV_ORDER if c in s.columns]).reset_index()
    s = s.melt(id_vars=factor, var_name="Severity", value_name="pct")

    # chart 2: stacked Bar chart
    stack = px.bar(s, x=factor, y="pct", color="Severity", color_discrete_map=sev_map(),
                   category_orders={"Severity": SEV_ORDER})
    stack.update_traces(marker_line_color=t["surface"], marker_line_width=1,
                        hovertemplate="%{x}<br>%{fullData.name}: %{y:.1f}%<extra></extra>")
    stack = style(stack, f"Severity mix of accidents by {factor.replace('_', ' ').lower()}", None, "% of accidents")

    # chart 3: Sunburst chart
    sun = px.sunburst(acc, path=["Weather", "Road_Condition", "Accident_Severity"],
                      color="Accident_Severity", color_discrete_map=sev_map())
    sun.update_traces(marker_line_color=t["surface"], insidetextfont_color=t["ink"],
                      hovertemplate="%{label}<br>%{value:,} accidents<extra></extra>")
    sun = style(sun, "Accidents: weather → road condition → severity (click to zoom)", legend=False)
    return mult, stack, sun

**Chart 1: Risk multiplier (horizontal bar)**
```python
base_cat = "Clear" if factor == "Weather" else "Dry"
r = dff.groupby(factor)["Accident_Occurred"].mean()
m = (r / base).sort_values()...
```
- Calculates each condition's accident rate and **divides it by the baseline** (Clear weather or Dry road).
- Result: a **risk multiple**, e.g. Snow = **1.70×** means 70% more likely than clear weather.
- `add_vline(x=1)` draws a dashed line at **1.0× (the baseline)**.
- `custom_data=["rate"]` passes the actual rate into the tooltip: *"Snow: 1.70× baseline, Rate 5.93%"*.

**Chart 2: Severity mix (100% stacked bar)**
```python
s = pd.crosstab(acc[factor], acc["Accident_Severity"], normalize="index").mul(100)
s = s.melt(...)
```
- `pd.crosstab(..., normalize="index")` builds a table of **severity percentages per condition** (each row adds up to 100%).
- `.melt()` reshapes it into the "long" format Plotly needs (one row per condition + severity).
- Colours come from `sev_map()`, ordered Minor → Fatal.

**Chart 3: Sunburst**
```python
px.sunburst(acc, path=["Weather", "Road_Condition", "Accident_Severity"], ...)
```
- A **hierarchical** chart: inner ring = weather → middle ring = road condition → outer ring = severity.
- Ring sizes are proportional to the **number of accidents**. Users can **click a section to zoom in**.

#### 9.6 Tab 3: Congestion vs speeding: `t3`

**Answers:** *Which factors best predict accidents? Does congestion, not speeding, drive accidents?*


In [43]:
# ---- Tab 3: congestion vs speeding ----
@app.callback([Output("cong-bar", "figure"), Output("speed-paradox", "figure"),
               Output("age-bar", "figure"), Output("exp-bar", "figure"), Output("corr", "figure")],
              FILTERS + [Input("f-mode", "value"), Input("t3-control", "value"), THEME])
def t3(years, areas, roads, weathers, mode, control, theme):
    use_theme(theme)
    t = P()
    dff = filter_df(years, areas, roads, weathers)
    if dff.empty:
        return [empty_fig()] * 5
    lab = metric_label(mode)
    fmt = ".2f" if mode == "rate" else ","

    def simple_bar(col, title, color):
        m = metric(dff, col, mode)
        f = px.bar(m, x=col, y="value", color_discrete_sequence=[color], text_auto=fmt)
        f.update_traces(marker_cornerradius=4, textposition="outside")
        return fit_labels(style(f, title, None, lab, legend=False))

    cong = simple_bar("Congestion_Band", f"{lab} by congestion level", t["series"][0])
    age = simple_bar("Age_Group", f"{lab} by driver age group", t["series"][2])
    exp = simple_bar("Experience_Band", f"{lab} by driving experience", t["series"][2])

    pair = [t["series"][1], t["series"][0]]
    if "on" in (control or []):
        m = metric(dff, ["Congestion_Band", "Speed_Status"], mode)
        sp = px.bar(m, x="Congestion_Band", y="value", color="Speed_Status", barmode="group",
                    color_discrete_sequence=pair, text_auto=fmt)
        title = "Speeding vs within limit, <b>within each congestion level</b>"
    else:
        m = metric(dff, "Speed_Status", mode)
        sp = px.bar(m, x="Speed_Status", y="value", color="Speed_Status",
                    color_discrete_sequence=pair, text_auto=fmt)
        title = "The speed paradox: is speeding really safer?"
    sp.update_traces(marker_cornerradius=4, textposition="outside")
    sp = fit_labels(style(sp, title, None, lab))

    num = ["Accident_Occurred", "Congestion_Index", "Recorded_Speed_kmh", "Speed_Limit_kmh",
           "Visibility_m", "Driver_Age", "Driver_Experience_yrs", "Vehicle_Age_yrs", "Hour", "Is_Holiday"]
    c = dff[num].corr().round(2)
    div = [[0, t["series"][0]], [0.5, t["div_mid"]], [1, t["series"][1]]]
    corr = px.imshow(c, text_auto=True, color_continuous_scale=div, zmin=-1, zmax=1, aspect="auto")
    corr = style(corr, "Correlation between numeric factors", legend=False)
    return cong, sp, age, exp, corr

**Answers:** *Which factors best predict accidents? Does congestion, not speeding, drive accidents?*

**`simple_bar()`: a helper defined inside the callback**
```python
def simple_bar(col, title, color):
    m = metric(dff, col, mode)
    ...
```
Builds a standard bar chart for any column. It's reused for **three charts**: accident rate by **congestion level**, by **driver age group**, and by **driving experience**.

**The speed paradox chart**
```python
if "on" in (control or []):
    m = metric(dff, ["Congestion_Band", "Speed_Status"], mode)   # split by congestion
else:
    m = metric(dff, "Speed_Status", mode)                         # simple comparison
```
- **Checkbox off:** compares **Over limit vs Within limit** directly. The result looks surprising: speeders appear *safer*.
- **Checkbox on ("Control for congestion"):** compares speeders and non-speeders **within each congestion level** (`barmode="group"`). This reveals the paradox is mostly a **congestion effect**: you can't speed in a traffic jam, and jams are where most accidents happen.
- `(control or [])` protects against the checkbox value being `None`.

**Correlation heatmap**
```python
c = dff[num].corr().round(2)
div = [[0, blue], [0.5, grey], [1, orange]]
corr = px.imshow(c, text_auto=True, color_continuous_scale=div, zmin=-1, zmax=1)
```
- `.corr()` calculates the **correlation** between each pair of numeric columns (-1 to +1).
- A **diverging colour scale**: **blue = negative**, **grey = none (0)**, **orange = positive**.
- `zmin=-1, zmax=1` keeps the colour scale fixed, so 0 is always grey.
- `text_auto=True` prints the value in each cell.

#### 9.7 Tab 4: Emergency response: `t4`

**Answers:** *Is emergency response meeting expectations?* Only accident rows are used (`Accident_Occurred == 1`), because only accidents have a response.

In [44]:
# ---- Tab 4: emergency response ----
@app.callback([Output("gap-box", "figure"), Output("gap-weather", "figure"), Output("gap-month", "figure")],
              FILTERS + [Input("t4-dim", "value"), THEME])
def t4(years, areas, roads, weathers, dim, theme):
    use_theme(theme)
    t = P()
    acc = filter_df(years, areas, roads, weathers)
    acc = acc[acc["Accident_Occurred"] == 1]
    if acc.empty:
        return empty_fig(), empty_fig(), empty_fig()

    order = acc.groupby(dim)["Response_Gap_min"].median().sort_values(ascending=False).index.tolist()

    # Chart 1: Response gap box plot
    box = px.box(acc, x=dim, y="Response_Gap_min", category_orders={dim: order},
                 color_discrete_sequence=[t["series"][0]], points=False)
    box.add_hline(y=0, line_color=t["muted"], line_dash="dash", annotation_text="On baseline",
                  annotation_font_color=t["muted"])
    box = style(box, f"Response gap by {dim.replace('_', ' ').lower()}", None, "Minutes over baseline", legend=False)

    w = acc.groupby("Weather")["Response_Gap_min"].mean().sort_values().reset_index()
    worst = w["Weather"].iloc[-1]

    # Chart 2: Average gap by weather (horizontal bar)
    wf = px.bar(w, x="Response_Gap_min", y="Weather", orientation="h",
                text=w["Response_Gap_min"].map("{:.1f} min".format),
                color=np.where(w["Weather"] == worst, "Worst", "Other"),
                color_discrete_map={"Worst": t["critical"], "Other": t["soft_bar"]})
    wf.update_traces(marker_cornerradius=4, textposition="outside")
    wf = fit_labels(style(wf, f"Average response gap by weather ({worst} is worst)",
                          "Minutes over baseline", None, legend=False), horizontal=True)

    mo = acc.groupby(["Year", "Month"])["Response_Gap_min"].mean().reset_index()
    mo["Year"] = mo["Year"].astype(str)

    # Chart 3: Monthly gap, year over year (smooth line chart)
    line = px.line(mo, x="Month", y="Response_Gap_min", color="Year", markers=True,
                   color_discrete_sequence=t["series"])
    line.update_traces(line=dict(shape="spline", smoothing=1.3, width=2.5), marker_size=6,
                       hovertemplate="%{x}: %{y:.1f} min<extra>%{fullData.name}</extra>")
    line.update_xaxes(tickvals=list(range(1, 13)), ticktext=MONTHS)
    line = style(line, "Monthly average response gap, year over year (smoothed)", None, "Minutes over baseline")
    return box, wf, line

**Chart 1: Response gap box plot**
```python
order = acc.groupby(dim)["Response_Gap_min"].median().sort_values(ascending=False).index.tolist()
box = px.box(acc, x=dim, y="Response_Gap_min", category_orders={dim: order}, points=False)
```
- `dim` comes from the breakdown dropdown (area type, weather, priority...).
- Groups are **sorted by median gap**, so the **worst group appears first**.
- A **box plot** shows the full spread: median, middle 50%, and extremes.
- `points=False` hides the thousands of individual dots for a cleaner chart.
- `add_hline(y=0)` marks **"On baseline"**. Anything above the line means help arrived **later than expected**.

**Chart 2: Average gap by weather (horizontal bar)**
```python
worst = w["Weather"].iloc[-1]
color=np.where(w["Weather"] == worst, "Worst", "Other")
color_discrete_map={"Worst": t["critical"], "Other": t["soft_bar"]}
```
- Calculates the average gap per weather type, sorted from lowest to highest.
- The **worst** weather (the last after sorting, Fog) is automatically coloured **red** and named in the title: *"(Fog is worst)"*.

**Chart 3: Monthly gap, year over year (smooth line chart)**
- Average gap per **year and month**; one line per year.
- `astype(str)` turns years into text so Plotly treats them as **separate categories** (one colour per year) instead of a continuous number scale.
- `shape="spline"` draws **smooth curves**.
- `tickvals` / `ticktext` replace month numbers (1–12) with names (Jan–Dec).

#### 9.8 Tab 5: Closures and disruption: `t5`
**Answers:** *How do severity and collision type drive road closure time?*

In [45]:
# ---- Tab 5: closures ----
@app.callback([Output("closure-box", "figure"), Output("collision-pie", "figure"),
               Output("treemap", "figure"), Output("top10", "data"), Output("top10", "columns")],
              FILTERS + [Input("t5-log", "value"), THEME])
def t5(years, areas, roads, weathers, log, theme):
    use_theme(theme)
    t = P()
    acc = filter_df(years, areas, roads, weathers)
    acc = acc[acc["Accident_Occurred"] == 1]
    if acc.empty:
        return empty_fig(), empty_fig(), empty_fig(), [], []

    box = px.box(acc, x="Accident_Severity", y="Closure_Hours", color="Accident_Severity",
                 category_orders={"Accident_Severity": SEV_ORDER}, color_discrete_map=sev_map(),
                 log_y="log" in (log or []))
    box = style(box, "Road closure duration by severity", None, "Closure (hours)", legend=False)

    pie_d = acc["Collision_Type"].value_counts().reset_index()
    pie = px.pie(pie_d, names="Collision_Type", values="count", hole=0.5,
                 color_discrete_sequence=t["series"])
    pie.update_traces(textinfo="percent+label", marker_line_color=t["surface"], marker_line_width=2,
                      outsidetextfont_color=t["ink2"])
    pie = style(pie, "Share of accidents by collision type", legend=False)

    tree = px.treemap(acc, path=["Road_Type", "Accident_Severity"], values="Closure_Hours",
                      color="Accident_Severity", color_discrete_map=sev_map())
    tree.update_traces(marker_line_color=t["surface"],
                       hovertemplate="%{label}<br>%{value:,.0f} closure hours<extra></extra>")
    tree = style(tree, "Total closure hours: road type → severity", legend=False)

    cols = ["Date", "Road_Type", "Area_Type", "Weather", "Accident_Severity", "Collision_Type", "Closure_Hours"]
    top = acc.nlargest(10, "Closure_Hours")[cols].copy()
    top["Date"] = top["Date"].dt.strftime("%Y-%m-%d")
    top["Closure_Hours"] = top["Closure_Hours"].round(1)
    return box, pie, tree, top.to_dict("records"), [{"name": c.replace("_", " "), "id": c} for c in cols]

**Chart 1: Closure duration by severity (box plot)**
- Shows how closure hours spread for each severity level, ordered Minor → Fatal.
- `log_y="log" in (log or [])` switches the y-axis to a **logarithmic scale** when the checkbox is ticked. This helps because closure times have a **long tail** (max ~29.5 h), which squashes the smaller boxes on a normal scale.

**Chart 2: Collision type share (donut chart)**
```python
pie_d = acc["Collision_Type"].value_counts().reset_index()
pie = px.pie(pie_d, names="Collision_Type", values="count", hole=0.5)
```
- `value_counts()` counts accidents per collision type.
- `hole=0.5` turns the pie into a **donut** (easier to read).
- `textinfo="percent+label"` shows e.g. *"Rear-End 59.6%"* on each slice.

**Chart 3: Treemap**
```python
px.treemap(acc, path=["Road_Type", "Accident_Severity"], values="Closure_Hours", ...)
```
- Rectangles sized by **total closure hours**: road type → severity inside.
- Shows **which segments cause the most disruption** at a glance.

**Table: Top 10 longest closures**
```python
top = acc.nlargest(10, "Closure_Hours")[cols].copy()
top["Date"] = top["Date"].dt.strftime("%Y-%m-%d")
return ..., top.to_dict("records"), [{"name": c.replace("_", " "), "id": c} for c in cols]
```
- `nlargest(10, ...)` picks the **10 longest closures**, the outliers.
- Dates are formatted as text (`2023-12-26`) and hours rounded to 1 decimal.
- `DataTable` needs two things:
  - `data`: the rows, as a list of dictionaries (`to_dict("records")`)
  - `columns`: the column headers, with underscores replaced by spaces ("Closure_Hours" → "Closure Hours")

**Summary:**
| Callback | Updates | Triggered by |
|---|---|---|
| `flip_theme`, `set_theme` | Theme store + page class | Toggle button |
| `update_kpis` | 5 KPI cards + YoY | Filters |
| `t1_heatmap` | Hour × day heatmap | Filters, Rate/Count, theme |
| `clear_click`, `t1_charts` | Line + 2 bar charts (drill-down) | Filters, heatmap click, clear button, theme |
| `t2` | Risk multiplier, severity mix, sunburst | Filters, Weather/Road switch, theme |
| `t3` | Congestion, speed paradox, age, experience, correlation | Filters, Rate/Count, checkbox, theme |
| `t4` | Response gap box, weather bar, monthly line | Filters, breakdown dropdown, theme |
| `t5` | Closure box, donut, treemap, Top-10 table | Filters, log-scale checkbox, theme |

Every chart listens to the **same four filters**, so the whole dashboard always stays in sync with the user's selection.

### Step 10: Export Feature and Running the App

This last cell adds the **"Download filtered data"** feature and starts the dashboard.

#### 10.1 Export the filtered data as a CSV: `download`


In [46]:
# ---- Export ----
@app.callback(Output("download", "data"), Input("btn-download", "n_clicks"),
              [State("f-year", "value"), State("f-area", "value"),
               State("f-road", "value"), State("f-weather", "value")], prevent_initial_call=True)
def download(_, years, areas, roads, weathers):
    dff = filter_df(years, areas, roads, weathers)
    return dcc.send_data_frame(dff.to_csv, "traffic_filtered.csv", index=False)

This lets the user **download exactly the data they're looking at** as a CSV file they can open in Excel.

| Part | Meaning |
|---|---|
| `Output("download", "data")` | Sends the file to the invisible `dcc.Download` component (created in the layout), which triggers the browser download |
| `Input("btn-download", "n_clicks")` | The callback runs **only when the button is clicked** |
| `State(...)` × 4 | Reads the current **Year, Area, Road and Weather** filter values |
| `prevent_initial_call=True` | Stops a download from starting automatically when the page first loads |
| `_` | The click count isn't needed, only the fact that a click happened |

**Why `State` and not `Input` for the filters?** With `Input`, a download would start **every time a filter changed**. With `State`, the filters are only **read at the moment the button is clicked**, which is the behaviour we want.

**How it works:**
1. `filter_df(...)` applies the same filters the charts use, so the file matches what's on screen.
2. `dcc.send_data_frame(dff.to_csv, "traffic_filtered.csv", index=False)` converts the table to CSV and sends it to the browser:
   - `dff.to_csv` is the conversion method (passed **without brackets**, so Dash calls it itself)
   - `"traffic_filtered.csv"` is the downloaded file name
   - `index=False` leaves out pandas' row numbers, so the file only contains the real data columns

**Example:** select *Year = 2024* and *Weather = Fog* → click Download → you get a CSV containing **only foggy 2024 records**. The file is created **in memory** and sent directly, so nothing is saved on the server.

This covers the project's **"Export dataset feature"** bonus requirement.

#### 10.2 Start the dashboard

In [47]:
if __name__ == "__main__":
    app.run(debug=True)

<IPython.core.display.Javascript object>

- `if __name__ == "__main__":` means "only run this when the file is **started directly**", e.g. with `python app.py`. If the file is **imported** by another program (such as a hosting server using the `server` variable from Step 6), the app won't start a second time.
- `app.run()` starts a **local web server** and prints a link, usually **http://127.0.0.1:8050/**. Open it in a browser to use the dashboard.
- `debug=True` turns on **development mode**:
  - **Auto-reload:** the app restarts automatically when the code is saved
  - **Error panel:** shows errors in the browser (the small panel in the bottom-right corner)
  - It should be switched **off** (`debug=False`) if the dashboard is ever published publicly.

**To run the dashboard:**
```
python app.py
```
Then open **http://127.0.0.1:8050/** in the browser. Press **Ctrl + C** in the terminal to stop it.

---

**Summary:**
| Part | Job |
|---|---|
| `download` callback | Exports the currently filtered data as a CSV when the button is clicked |
| `app.run(debug=True)` | Starts the dashboard on a local web server with auto-reload and error display |

This completes the dashboard: **data preparation → theme and styling → layout → interactive callbacks → export → run**.